# 01 — Data Preparation & RFM Feature Engineering

**Customer Lifetime Value (CLV) Analysis — Online Retail II**

This notebook cleans the raw transaction data, engineers RFM (Recency, Frequency,
Monetary) features, and segments customers with K-Means clustering.

---

### A note on this notebook's history

An earlier version of this notebook had a real bug: a column-normalisation step
silently dropped the `InvoiceDate` column, which caused `recency` to be populated
with **transaction row-counts instead of days since last purchase** (max recency
came out to 12,890 "days" on a dataset spanning only 739 days — impossible).

That bug is explained and fixed in the **Data Cleaning** section below, rather
than hidden. The root cause, the fix, and an assertion that makes the same class
of bug impossible to reintroduce silently are all shown inline.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../Data")
OUT_DIR = Path("Outputs")
OUT_DIR.mkdir(exist_ok=True)

## 1. Load raw data

Online Retail II (UCI / Kaggle): ~1M transaction line items across two sheets
(2009–2010 and 2010–2011).

In [2]:
excel_path = DATA_DIR / "online_retail_II.xlsx"
xls = pd.ExcelFile(excel_path)
raw = pd.concat([xls.parse(s) for s in xls.sheet_names], ignore_index=True)

print("Sheets:", xls.sheet_names)
print("Loaded:", raw.shape)
raw.head()

Sheets: ['Year 2009-2010', 'Year 2010-2011']
Loaded: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 2. Column normalisation — and the bug that lived here

The original version of this cell normalised column names like this:

```python
raw.columns = (raw.columns.str.strip().str.lower()
               .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
```

That regex only inserts an underscore where a **non-alphanumeric character already
exists**. `"Customer ID"` has a space, so it correctly became `customer_id`. But
`"InvoiceDate"` has no separator at all — no space, no underscore — so it became
`invoicedate`, **not** `invoice_date`.

A later step filtered the dataframe down to a hard-coded list of expected column
names, one of which was `invoice_date`. Since that exact string didn't exist
anymore, **the date column was silently dropped**. The RFM aggregation had a
fallback for exactly this situation:

```python
recency = ("invoice_date", lambda s: (ref_date - s.max()).days) \
          if "invoice_date" in raw.columns else ("customer_id", "size")
```

The fallback fired. `recency` was quietly populated with each customer's
**transaction row-count**, not days since last purchase. Nothing raised an
error — the pipeline just produced plausible-looking, wrong numbers.

**The fix:** split camelCase into snake_case *before* lowercasing, so
`InvoiceDate` → `invoice_date` correctly. An assertion below also makes any
future column-name mismatch fail loudly instead of silently substituting a
different variable.

In [3]:
raw.columns = (
    raw.columns.str.strip()
    .str.replace(r"(?<=[a-z0-9])(?=[A-Z])", "_", regex=True)  # camelCase -> snake_case
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
print("Columns after normalisation:", list(raw.columns))

required = {"invoice", "invoice_date", "quantity", "price", "customer_id"}
missing = required - set(raw.columns)
assert not missing, f"Required columns missing after normalisation: {missing}"
print("All required columns present.")

Columns after normalisation: ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'price', 'customer_id', 'country']
All required columns present.


## 3. Clean

Drop rows with missing keys, coerce types, and remove cancelled orders / non-positive
quantities and prices (returns and data-entry errors).

In [4]:
raw["invoice_date"] = pd.to_datetime(raw["invoice_date"], errors="coerce")
for c in ("quantity", "price"):
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

raw = raw.dropna(subset=["invoice", "invoice_date", "quantity", "price", "customer_id"])
raw = raw[(raw["quantity"] > 0) & (raw["price"] > 0)]
raw["amount"] = raw["quantity"] * raw["price"]

print("Cleaned:", raw.shape)
print("Date range:", raw["invoice_date"].min(), "->", raw["invoice_date"].max())

Cleaned: (805549, 9)
Date range: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00


## 4. Build RFM features

- **Recency** — days since the customer's most recent purchase (relative to one day
  after the dataset's last transaction)
- **Frequency** — number of distinct invoices (orders)
- **Monetary** — total amount spent

The assertion below is the direct fix for the original bug: **recency cannot
exceed the span of the dataset.** If it ever does again, this notebook will fail
immediately here instead of producing silently wrong numbers downstream.

In [5]:
ref_date = raw["invoice_date"].max() + pd.Timedelta(days=1)
print("Reference date:", ref_date)

rfm = (
    raw.groupby("customer_id")
    .agg(
        recency=("invoice_date", lambda s: (ref_date - s.max()).days),
        frequency=("invoice", "nunique"),
        monetary=("amount", "sum"),
    )
    .reset_index()
)
rfm["customer_id"] = rfm["customer_id"].astype(int)

span = (raw["invoice_date"].max() - raw["invoice_date"].min()).days + 1
assert rfm["recency"].max() <= span, (
    f"Recency max {rfm['recency'].max()} exceeds dataset span {span} days -- "
    "date handling is broken."
)

print(f"Recency range: {rfm['recency'].min()}-{rfm['recency'].max()} days (span {span})")
print(f"Customers: {len(rfm):,} | Revenue: ${rfm['monetary'].sum():,.0f}")
rfm.describe()

Reference date: 2011-12-10 12:50:00


Recency range: 1-739 days (span 739)
Customers: 5,878 | Revenue: $17,743,429


,customer_id,recency,frequency,monetary
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,201.331916,6.289384,3018.616737
std,1715.572666,209.338707,13.009406,14737.731040
min,12346.000000,1.000000,1.000000,2.950000
25%,13833.250000,26.000000,1.000000,348.762500
50%,15314.500000,96.000000,3.000000,898.915000
75%,16797.750000,380.000000,7.000000,2307.090000
max,18287.000000,739.000000,398.000000,608821.650000


## 5. K-Means segmentation

RFM values are heavily right-skewed, so features are **log-transformed before
scaling**. Clustering on raw scaled values scores a technically higher silhouette,
but only because K-Means isolates a handful of extreme outliers into their own
tiny cluster — not a usable business segment. log1p trades that metric for
segments a marketing team can actually act on (documented with numbers in
`METHODOLOGY.md`).

`k=3` was chosen by comparing silhouette score and Calinski-Harabasz across
`k=2..6` and checking that no resulting cluster was too small to be useful.

In [6]:
FEATURES = ["recency", "frequency", "monetary"]

X = StandardScaler().fit_transform(np.log1p(rfm[FEATURES].values))
rfm["cluster"] = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X)

# Name clusters by ranked median spend rather than by hand, so re-running this
# notebook can never silently reassign a segment's name to the wrong cluster id.
profile = rfm.groupby("cluster")["monetary"].median().sort_values()
ordered = list(profile.index)
SEGMENTS = {ordered[0]: "Hibernating", ordered[1]: "Loyal Customers", ordered[2]: "Champions"}
rfm["segment"] = rfm["cluster"].map(SEGMENTS)

rfm[["customer_id"] + FEATURES + ["cluster"]].to_csv(OUT_DIR / "rfm_with_clusters.csv", index=False)
rfm[["customer_id"] + FEATURES + ["cluster", "segment"]].to_csv(
    OUT_DIR / "rfm_with_clusters_and_segments.csv", index=False
)
rfm.head()

,customer_id,recency,frequency,monetary,cluster,segment
0,12346,326,12,77556.46,2,Champions
1,12347,2,8,5633.32,2,Champions
2,12348,75,5,2019.40,0,Loyal Customers
3,12349,19,4,4428.69,0,Loyal Customers
4,12350,310,1,334.40,1,Hibernating


In [7]:
total = rfm["monetary"].sum()
summary = (
    rfm.groupby("cluster")
    .agg(
        customers=("customer_id", "size"),
        median_recency=("recency", "median"),
        median_frequency=("frequency", "median"),
        median_monetary=("monetary", "median"),
        total_revenue=("monetary", "sum"),
    )
    .reset_index()
)
summary["segment"] = summary["cluster"].map(SEGMENTS)
summary["pct_customers"] = (summary["customers"] / len(rfm) * 100).round(1)
summary["pct_revenue"] = (summary["total_revenue"] / total * 100).round(1)
summary.to_csv(OUT_DIR / "cluster_summary.csv", index=False)

summary[["segment", "customers", "pct_customers", "pct_revenue"]]

,segment,customers,pct_customers,pct_revenue
0,Loyal Customers,2268,38.6,19.9
1,Hibernating,2397,40.8,5.4
2,Champions,1213,20.6,74.7


**Cross-check against the headline finding:** the Champions segment
(top-value cluster) should hold roughly the same revenue share as sorting all
customers by spend and taking the top 20% — two independent methods, one
finding. That agreement is good evidence the segmentation is real, not an
artifact of clustering choices.

In [8]:
s = rfm.sort_values("monetary", ascending=False)
top20_share = s.head(int(round(len(rfm) * 0.20)))["monetary"].sum() / total
champions_share = summary.loc[summary["segment"] == "Champions", "pct_revenue"].iloc[0] / 100

print(f"Top 20% of customers by spend hold {top20_share:.1%} of revenue")
print(f"Champions cluster holds            {champions_share:.1%} of revenue")

Top 20% of customers by spend hold 77.3% of revenue
Champions cluster holds            74.7% of revenue


## 6. Marketing cohorts

Each cohort is generated from an explicit, auditable rule rather than a hand-picked
list, so the exported CSV can never silently drift from what its filename claims.

In [9]:
med_mon = rfm["monetary"].median()
lapse = rfm["recency"].quantile(0.60)

cohorts = {
    "targets_vip": rfm.nlargest(int(round(len(rfm) * 0.20)), "monetary"),
    "targets_core": rfm[rfm["segment"] == "Champions"],
    "targets_atrisk": rfm[(rfm["monetary"] > med_mon) & (rfm["recency"] >= lapse)],
}
for name, df in cohorts.items():
    df[["customer_id"] + FEATURES].to_csv(OUT_DIR / f"{name}.csv", index=False)
    print(f"{name:15s} n={len(df):5d}  revenue {df['monetary'].sum()/total*100:5.1f}%")

targets_vip     n= 1176  revenue  77.3%
targets_core    n= 1213  revenue  74.7%
targets_atrisk  n=  622  revenue   9.6%


## 7. CLV modelling datasets (temporal holdout)

Written here so the modelling notebook never has to re-read the raw Excel file,
and so the calibration/holdout split is defined in exactly one place.

- **Calibration window** — features are built from purchase history up to the cutoff
- **Holdout window** — the target is what the customer actually spent *after* the cutoff

This is what makes the downstream model a genuine forecast rather than a
same-period correlation.

In [10]:
CUTOFF = pd.Timestamp("2011-06-09")  # ~18 months calibration, ~6 months holdout

def build_features(df, ref):
    g = df.groupby("customer_id")
    f = pd.DataFrame({
        "recency": (ref - g.invoice_date.max()).dt.days,
        "frequency": g.invoice.nunique(),
        "monetary": g.amount.sum(),
        "tenure": (ref - g.invoice_date.min()).dt.days,
        "avg_order_value": g.amount.sum() / g.invoice.nunique(),
        "n_items": g.quantity.sum(),
        "n_products": g.stock_code.nunique(),
    }).reset_index()
    f["purchase_rate"] = f.frequency / f.tenure.clip(lower=1) * 30
    f["avg_gap"] = f.tenure / f.frequency.clip(lower=1)
    return f

calib = raw[raw.invoice_date <= CUTOFF]
hold = raw[raw.invoice_date > CUTOFF]

train = build_features(calib, CUTOFF + pd.Timedelta(days=1))
train = train.merge(
    hold.groupby("customer_id").amount.sum().rename("future_value"),
    on="customer_id", how="left",
).fillna({"future_value": 0.0})
train.to_csv(OUT_DIR / "clv_training_data.csv", index=False)

score = build_features(raw, raw.invoice_date.max() + pd.Timedelta(days=1))
score.to_csv(OUT_DIR / "clv_scoring_features.csv", index=False)

print(f"clv_training_data    n={len(train):,} | returned in holdout: {(train.future_value>0).mean():.1%}")
print(f"clv_scoring_features n={len(score):,}")

clv_training_data    n=4,966 | returned in holdout: 52.0%
clv_scoring_features n=5,878


## Summary

| Output | Rows | Notes |
|---|---|---|
| `rfm_with_clusters_and_segments.csv` | ~5,878 | Full customer base with segment labels |
| `cluster_summary.csv` | 3 | Segment-level revenue concentration |
| `targets_vip.csv`, `targets_core.csv`, `targets_atrisk.csv` | varies | Explicit-rule marketing cohorts |
| `clv_training_data.csv` | ~4,966 | Calibration features + holdout target, for model training |
| `clv_scoring_features.csv` | ~5,878 | Full-history features, for scoring every customer |

Next notebook: **`02_model_training_xgboost.ipynb`** trains the CLV model on
`clv_training_data.csv`.